# Simulating Constitutive Processes of semantic change within heterogeneous populations of speakers

In [1]:
# basic imports
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import statsmodels.formula.api as smf

# project code imports
from mod.one_hot_agent import *
from mod.plot import *
from mod.network import *

##### Hyper parameters

In [2]:
model_version = 'dy-1hot'

In [3]:
turns = 300
no_agents = 2
no_connections = 1
add_vocab_in = .98
semantic_features = 3
starting_observations = 5
starting_uncertainty = .2
words_per_semantic_feature = 100
new_environment_prob = .25
enforce_word_feature_mapping = False
no_simulations = 100

In [4]:
model_path = os.path.join('html',model_version)
if not os.path.exists(model_path):
    os.mkdir(model_path)

Simplifying the way episodes are run :)

In [5]:
def episode(
        net,
        starting_env,
        new_environment_prob: None|float=None,
        new_vocab_round_prob: None|float=None,
        stochastic_environment_updates: bool=False,
        lexicon_smoothening: float=1.
):

    if new_environment_prob:
        new_env_prob = torch.rand(size=(1,))
        if new_env_prob > new_environment_prob:
            starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

    new_vocab_round = False
    if new_vocab_round_prob:
        new_vocab_prob = torch.rand(size=(1,))
        if new_vocab_prob > new_vocab_round_prob:
            new_vocab_round = True


    env = starting_env.sample()
    if stochastic_environment_updates:
        starting_env.loc = env

    round_feature = torch.zeros(size=env.shape)
    round_feature[:,np.random.choice(env.shape[-1])] = 1.
    env = round_feature * env

    net.interaction(env, lexicon_smoothening, new_vocab_round)

    return starting_env, net

## Randomly generated environment

In [6]:
env_name = 'Randomly-Generated-Environment'

In [7]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [8]:
vocab_dif, H_dif = [], []

In [9]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(net, starting_env)

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [10]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     29.45
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           5.77e-08
Time:                        17:00:37   Log-Likelihood:            -2.7323e+05
No. Observations:               60000   AIC:                         5.465e+05
Df Residuals:                   59998   BIC:                         5.465e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        297.0202      0.188   1578.

In [12]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment,297.020219,0.188167,1578.491116,0.000000e+00
interaction_no,Randomly-Generated-Environment,-0.005880,0.001084,-5.426341,5.774307e-08


In [13]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [14]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     166.1
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           6.02e-38
Time:                        17:00:56   Log-Likelihood:                -62318.
No. Observations:               60000   AIC:                         1.246e+05
Df Residuals:                   59998   BIC:                         1.247e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.3153      0.006   1.56e

In [15]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment,873.315271,0.005596,156057.966453,0.000000e+00
interaction_no,Randomly-Generated-Environment,-0.000415,0.000032,-12.886530,6.019890e-38


In [16]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [17]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [18]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,301.659418
1,1,2,301.364389
2,1,3,301.099701
3,1,4,300.772294
4,1,5,300.499524


In [19]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [20]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [21]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [22]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Random environment and introduction of new terms

In [23]:
env_name = 'Randomly-Generated-Environment-and-Introducing-Novel-Terms'

In [24]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [25]:
vocab_dif, H_dif = [], []

In [26]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [27]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     30.31
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           3.70e-08
Time:                        17:01:36   Log-Likelihood:            -2.7400e+05
No. Observations:               60000   AIC:                         5.480e+05
Df Residuals:                   59998   BIC:                         5.480e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        297.0651      0.191   1558.

In [28]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment-and-Introducing...,297.065143,0.190596,1558.610491,0.000000e+00
interaction_no,Randomly-Generated-Environment-and-Introducing...,-0.006043,0.001098,-5.505378,3.698807e-08


In [29]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [30]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.420
Model:                            OLS   Adj. R-squared:                  0.420
Method:                 Least Squares   F-statistic:                 4.347e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        17:01:51   Log-Likelihood:            -1.3350e+05
No. Observations:               60000   AIC:                         2.670e+05
Df Residuals:                   59998   BIC:                         2.670e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.7236      0.018   4.77e

In [31]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment-and-Introducing...,873.723649,0.018327,47674.307750,0.0
interaction_no,Randomly-Generated-Environment-and-Introducing...,0.022007,0.000106,208.500925,0.0


In [32]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [33]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [34]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,301.636009
1,1,2,301.298275
2,1,3,301.044878
3,1,4,300.699875
4,1,5,300.479489


In [35]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [36]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [37]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [38]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Changing/Upheaval environment

In [39]:
env_name = 'Suddenly-Changing-Environment'

In [40]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [41]:
vocab_dif, H_dif = [], []

In [42]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [43]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     32.64
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           1.11e-08
Time:                        17:02:29   Log-Likelihood:            -2.8041e+05
No. Observations:               60000   AIC:                         5.608e+05
Df Residuals:                   59998   BIC:                         5.608e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        294.8906      0.212   1390.

In [44]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment,294.890569,0.212063,1390.581726,0.000000e+00
interaction_no,Suddenly-Changing-Environment,-0.006978,0.001221,-5.713306,1.113250e-08


In [45]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [46]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.075
Method:                 Least Squares   F-statistic:                     4868.
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        17:02:29   Log-Likelihood:                -83994.
No. Observations:               60000   AIC:                         1.680e+05
Df Residuals:                   59998   BIC:                         1.680e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        874.5530      0.008   1.09e

In [47]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment,874.553012,0.008031,108894.799369,0.0
interaction_no,Suddenly-Changing-Environment,0.003227,0.000046,69.771966,0.0


In [48]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [49]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [50]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,299.884105
1,1,2,299.548108
2,1,3,299.268700
3,1,4,298.999257
4,1,5,298.736282


In [51]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [52]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [53]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [54]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Changing/Upheaval environment and introduction of new terms

In [ ]:
env_name = 'Suddenly-Changing-Environment-and-Introducing-Novel-Terms'

In [ ]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [ ]:
vocab_dif, H_dif = [], []

In [ ]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

#### Vocab difference stats

In [ ]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

In [ ]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

In [ ]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [ ]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

In [ ]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

In [ ]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [ ]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [ ]:
vocab_dif.head()

In [ ]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [ ]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [ ]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [ ]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Stochastic environment

In [55]:
env_name = 'Stochastically-Changing-Environment'

In [56]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [57]:
vocab_dif, H_dif = [], []

In [58]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

In [59]:
np.isinf(H_dif).sum()

simulation            0
agent                 0
interaction_no        0
delta             48330
dtype: int64

In [60]:
H_dif = H_dif.loc[~np.isinf(H_dif['delta'])]

#### Vocab difference stats

In [61]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     45.26
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           1.74e-11
Time:                        17:03:06   Log-Likelihood:            -2.7662e+05
No. Observations:               60000   AIC:                         5.532e+05
Df Residuals:                   59998   BIC:                         5.533e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        297.2707      0.199   1493.

In [62]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment,297.270709,0.199077,1493.243179,0.000000e+00
interaction_no,Stochastically-Changing-Environment,-0.007714,0.001147,-6.727904,1.736647e-11


In [63]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [64]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     103.1
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           4.01e-24
Time:                        17:03:17   Log-Likelihood:                -70037.
No. Observations:               11670   AIC:                         1.401e+05
Df Residuals:                   11668   BIC:                         1.401e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept       1017.4402      1.803    564.

In [65]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment,1017.440250,1.803107,564.270674,0.000000e+00
interaction_no,Stochastically-Changing-Environment,0.105352,0.010376,10.153794,4.014673e-24


In [66]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [67]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [68]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,302.837618
1,1,2,302.523955
2,1,3,302.259398
3,1,4,301.937950
4,1,5,301.656871


In [69]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [70]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [71]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [72]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Stochastic environment and introduction of new terms

In [73]:
env_name = 'Stochastically-Changing-Environment-and-Introducing-Novel-Terms'

In [74]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [75]:
vocab_dif, H_dif = [], []

In [76]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            stochastic_environment_updates=True,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

In [77]:
np.isinf(H_dif).sum()

simulation            0
agent                 0
interaction_no        0
delta             54486
dtype: int64

In [78]:
H_dif = H_dif.loc[~np.isinf(H_dif['delta'])]

#### Vocab difference stats

In [79]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     858.6
Date:                Tue, 10 Feb 2026   Prob (F-statistic):          2.12e-187
Time:                        17:03:45   Log-Likelihood:            -4.4052e+05
No. Observations:               60000   AIC:                         8.810e+05
Df Residuals:                   59998   BIC:                         8.811e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        334.2590      3.058    109.

In [80]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment-and-Introd...,334.258991,3.057525,109.323396,0.000000e+00
interaction_no,Stochastically-Changing-Environment-and-Introd...,0.515952,0.017609,29.301093,2.119820e-187


In [81]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [82]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.071
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     419.9
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           5.40e-90
Time:                        17:03:53   Log-Likelihood:                -30645.
No. Observations:                5514   AIC:                         6.129e+04
Df Residuals:                    5512   BIC:                         6.131e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        950.1715      1.638    579.

In [83]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment-and-Introd...,950.171460,1.638271,579.984269,0.000000e+00
interaction_no,Stochastically-Changing-Environment-and-Introd...,0.203337,0.009923,20.491346,5.397975e-90


In [84]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [85]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [86]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,298.509000
1,1,2,300.682292
2,1,3,322.437661
3,1,4,321.630211
4,1,5,313.933152


In [87]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [88]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [89]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [90]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Upheaval + stochastic environment

In [91]:
env_name = 'Suddenly-and-Stochastically-Changing-Environment'

In [92]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [93]:
vocab_dif, H_dif = [], []

In [94]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [95]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     40.17
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           2.35e-10
Time:                        17:04:23   Log-Likelihood:            -2.7144e+05
No. Observations:               60000   AIC:                         5.429e+05
Df Residuals:                   59998   BIC:                         5.429e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        293.2451      0.183   1605.

In [96]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environment,293.245057,0.182610,1605.855515,0.000000e+00
interaction_no,Suddenly-and-Stochastically-Changing-Environment,-0.006665,0.001052,-6.337863,2.346217e-10


In [97]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [98]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.065
Method:                 Least Squares   F-statistic:                     4160.
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        17:04:30   Log-Likelihood:                -92962.
No. Observations:               60000   AIC:                         1.859e+05
Df Residuals:                   59998   BIC:                         1.859e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        874.5964      0.009   9.38e

In [99]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environment,874.596409,0.009326,93780.818598,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environment,0.003464,0.000054,64.498772,0.0


In [100]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [101]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [102]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,298.138289
1,1,2,297.821948
2,1,3,297.561425
3,1,4,297.314593
4,1,5,297.042865


In [103]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [104]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [105]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [106]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Upheaval + stochastic environment, plus introduction of new terms

In [107]:
env_name = 'Suddenly-and-Stochastically-Changing-Environment-and-Introducing-Novel-Terms'

In [108]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [109]:
vocab_dif, H_dif = [], []

In [110]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            new_vocab_round_prob=add_vocab_in,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [111]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     21.72
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           3.16e-06
Time:                        17:04:52   Log-Likelihood:            -2.7982e+05
No. Observations:               60000   AIC:                         5.596e+05
Df Residuals:                   59998   BIC:                         5.597e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        293.7867      0.210   1399.

In [112]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environme...,293.786706,0.209989,1399.060427,0.000000
interaction_no,Suddenly-and-Stochastically-Changing-Environme...,-0.005636,0.001209,-4.660278,0.000003


In [113]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [114]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.542
Model:                            OLS   Adj. R-squared:                  0.542
Method:                 Least Squares   F-statistic:                 7.092e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        17:04:52   Log-Likelihood:            -1.3574e+05
No. Observations:               60000   AIC:                         2.715e+05
Df Residuals:                   59998   BIC:                         2.715e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        874.0710      0.019   4.59e

In [115]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environme...,874.070989,0.019025,45942.426448,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environme...,0.029178,0.000110,266.299641,0.0


In [116]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [117]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [118]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,297.790713
1,1,2,297.547272
2,1,3,297.322609
3,1,4,297.068751
4,1,5,296.859213


In [119]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [120]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [121]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [122]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)